# Valve Flow Characteristics

Exploring the flow behaviour of the `Valve` block under varying pressure drop conditions.
The valve models isenthalpic flow using a standard Cv equation:

$$F = C_v \sqrt{|\Delta P|} \cdot \text{sign}(\Delta P)$$

This example is inspired by [MiniSim's ValveExample](https://github.com/Nukleon84/MiniSim/blob/master/doc/ValveExample.ipynb),
adapted to PathSim's dynamic framework.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_chem.process import Valve

## Pressure Drop Sweep

Hold the outlet pressure at 1 bar and ramp the inlet pressure from 1 bar to 10 bar.
Compare valves with different Cv coefficients.

In [ ]:
results = {}

for Cv in [0.5, 1.0, 2.0, 5.0]:
    valve = Valve(Cv=Cv)

    P_in = Source(func=lambda t: 100000.0 + t * 10000.0)  # 1 bar -> 10 bar
    P_out = Source(func=lambda t: 100000.0)                 # 1 bar fixed
    T_in = Source(func=lambda t: 350.0)                     # 350 K

    scp = Scope(labels=["F", "T_out"])

    sim = Simulation(
        blocks=[P_in, P_out, T_in, valve, scp],
        connections=[
            Connection(P_in, valve),           # P_in -> port 0
            Connection(P_out, valve[1]),        # P_out -> port 1
            Connection(T_in, valve[2]),         # T_in -> port 2
            Connection(valve, scp),             # F -> scope 0
            Connection(valve[1], scp[1]),       # T_out -> scope 1
        ],
        dt=0.5,
    )

    sim.run(90)
    time, signals = scp.read()
    dP = (100000.0 + time * 10000.0) - 100000.0  # pressure drop [Pa]
    results[f"Cv={Cv}"] = (dP / 1e5, signals[0])  # convert dP to bar

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for label, (dP_bar, F) in results.items():
    ax.plot(dP_bar, F, label=label)

ax.set_xlabel(r"Pressure drop $\Delta P$ [bar]")
ax.set_ylabel("Flow rate $F$")
ax.set_title("Valve Characteristic Curves")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The square-root relationship between pressure drop and flow rate is clearly visible.
Doubling the Cv coefficient doubles the flow at the same pressure drop.

## Isenthalpic Expansion

The valve is an isenthalpic device — outlet temperature equals inlet temperature
regardless of pressure drop. Verify this across the full sweep.

In [ ]:
# Re-run one case and check T_out
valve = Valve(Cv=2.0)

P_in = Source(func=lambda t: 100000.0 + t * 10000.0)
P_out = Source(func=lambda t: 100000.0)
T_in = Source(func=lambda t: 350.0)

scp = Scope(labels=["F", "T_out"])

sim = Simulation(
    blocks=[P_in, P_out, T_in, valve, scp],
    connections=[
        Connection(P_in, valve),
        Connection(P_out, valve[1]),
        Connection(T_in, valve[2]),
        Connection(valve, scp),
        Connection(valve[1], scp[1]),
    ],
    dt=0.5,
)

sim.run(90)
time, signals = scp.read()

print(f"T_out range: {signals[1].min():.2f} K to {signals[1].max():.2f} K")
print(f"T_in = 350.00 K (constant) -> isenthalpic confirmed")